# Load and process Bacillus data

In [23]:
import numpy as np
import pandas as pd
from pathlib import Path
from Bio import SeqIO, Phylo, AlignIO
from Bio.Align import MultipleSeqAlignment
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import sys
sys.path.append('../pysimARG')
from fasta_to_bool import fasta_to_bool
from newick_to_tree import newick_to_tree
from segment_summary_stats import segment_summary_stats
from clonal_genealogy import ClonalTree

## Load `.nwk` tree

In [2]:
clonal_tree = Phylo.read("../data/bacillus/bacillus.nwk", "newick")
Phylo.draw_ascii(clonal_tree)

                                                                      ____ 1
                                                                   __|
                                                                 _|  |____ 4
                                                                | |
                                                           _____| |_______ 12
                                                          |     |
                                                          |     |       __ 2
                                          ________________|     |______|
                                         |                |            |__ 11
                                         |                |
              ___________________________|                |_______________ 10
             |                           |
             |                           |                            ____ 3
             |                           |             ______________|
      

In [3]:
edge, node_height = newick_to_tree(clonal_tree)

In [4]:
edge, node_height

(array([[14.      ,  1.      ,  0.104385],
        [14.      ,  4.      ,  0.104385],
        [15.      , 14.      ,  0.059258],
        [15.      , 12.      ,  0.163643],
        [16.      , 11.      ,  0.062855],
        [16.      ,  2.      ,  0.062855],
        [17.      , 16.      ,  0.152469],
        [17.      , 15.      ,  0.051681],
        [18.      , 10.      ,  0.339388],
        [18.      , 17.      ,  0.124064],
        [19.      ,  3.      ,  0.091161],
        [19.      ,  9.      ,  0.091161],
        [20.      , 19.      ,  0.321477],
        [20.      ,  5.      ,  0.412638],
        [21.      , 20.      ,  0.299199],
        [21.      , 18.      ,  0.372449],
        [22.      ,  7.      ,  0.188528],
        [22.      ,  6.      ,  0.188528],
        [23.      ,  8.      ,  0.564415],
        [23.      , 22.      ,  0.375887],
        [24.      , 21.      ,  0.606251],
        [24.      , 23.      ,  0.753673],
        [25.      , 24.      ,  0.261099],
        [25

In [5]:
node_height = node_height[~np.isnan(node_height)]
node_height

array([0.      , 0.      , 0.      , 0.      , 0.      , 0.      ,
       0.      , 0.      , 0.      , 0.      , 0.      , 0.      ,
       0.      , 0.104385, 0.163643, 0.062855, 0.215324, 0.339388,
       0.091161, 0.412638, 0.711837, 0.188528, 0.564415, 1.318088,
       1.579187])

In [6]:
# np.savetxt("../data/bacillus/clonal_edge.csv", edge, delimiter=",")
# np.savetxt("../data/bacillus/clonal_node_height.csv", node_height, delimiter=",")
# clonal_edge = np.loadtxt("../data/bacillus/clonal_edge.csv", delimiter=",", dtype=float)
# clonal_node_height = np.loadtxt("../data/bacillus/clonal_node_height.csv", delimiter=",", dtype=float)

## Load `.xmfa` data

In [ ]:
def read_xmfa_biopython(file_path):
    blocks = list(AlignIO.parse(file_path, "mauve"))
    return blocks

In [ ]:
def remove_gap_columns(alignment_block):
    """
    Takes a Biopython MultipleSeqAlignment block and returns a new block
    with any column containing a '-' completely removed.
    """
    align_len = alignment_block.get_alignment_length()
    
    valid_columns = [i for i in range(align_len) if "-" not in alignment_block[:, i]]

    cleaned_records = []
    for record in alignment_block:
        clean_seq_string = "".join([record.seq[i] for i in valid_columns])
        new_record = SeqRecord(Seq(clean_seq_string), id=record.id, description="")
        cleaned_records.append(new_record)

    return MultipleSeqAlignment(cleaned_records)

In [ ]:
xmfa_path = "../data/bacillus/bacillus.xmfa"
aligned_blocks = read_xmfa_biopython(xmfa_path)

print(f"Total blocks successfully loaded: {len(aligned_blocks)}")

Total blocks successfully loaded: 1218

--- First Block Details ---
Alignment length: 10511
Number of sequences in this block: 13
ID: Bacillus_anthracis_Ames_0581.gbk/71581-82075
Seq: tggtattaaaaattgaaatggagagtaatctttttgtaaaatgatgagga...
ID: Bacillus_cereus_03BB102.gbk/71476-81970
Seq: tggtattaaaaattgaaatggggagtaatctttttgtaaaatgatgagga...
ID: Bacillus_cereus_AH187.gbk/71598-82092
Seq: tggtactaaaaattgaaatagggagtaatctttttgtaaaatgatgagga...
ID: Bacillus_cereus_AH820.gbk/71591-82085
Seq: tggtattaaaaattgaaatggggagtaatctttttgtaaaatgatgagga...
ID: Bacillus_cereus_ATCC_10987.gbk/71588-82081
Seq: tggtattaaaaattgaaatagggagtaatctttttgtaaaatgatgagga...
ID: Bacillus_cereus_ATCC14579.gbk/71362-81867
Seq: tagtattaaaaattgaaatagagagcaatctttttgtaaaatgatgagga...
ID: Bacillus_cereus_B4264.gbk/71485-81990
Seq: tagtattaaaaattgaaatagagagcaatctttttgtaaaatgatgaggg...
ID: Bacillus_cereus_G9842.gbk/71329-81825
Seq: tagtattaaaaattgaaatagagagcaatctttttgtaaaatgatgagga...
ID: Bacillus_cereus_Q1.gbk/71369-81863
Seq: 

In [8]:
first_block = aligned_blocks[11]
print(f"\n--- 11th Block Details ---")
print(f"Alignment length: {first_block.get_alignment_length()}")
print(f"Number of sequences in this block: {len(first_block)}")


for record in first_block:
    print(f"ID: {record.id}")
    print(f"Seq: {record.seq[:50]}...")


--- 11th Block Details ---
Alignment length: 3719
Number of sequences in this block: 13
ID: Bacillus_anthracis_Ames_0581.gbk/155978-159689
Seq: aagaaactagccaagagccaacagtag------ataatcaaaaagagcca...
ID: Bacillus_cereus_03BB102.gbk/161566-165277
Seq: aagaaactagccaagagccaacagtag------ataatcaaaaagagcca...
ID: Bacillus_cereus_AH187.gbk/161744-165454
Seq: aagaaactagccaagagccaacagtag------ataatcaaaaagagcca...
ID: Bacillus_cereus_AH820.gbk/155987-159698
Seq: aagaaactagccaagagccaacagtag------ataatcaaaaagagcca...
ID: Bacillus_cereus_ATCC_10987.gbk/155981-159691
Seq: aagaaactagccaagagccaacagtag------ataatcaaaaagagcca...
ID: Bacillus_cereus_ATCC14579.gbk/161626-165338
Seq: aagaaacgagtcaagaaccaacagttg------ataatcaaaaagagcaa...
ID: Bacillus_cereus_B4264.gbk/161751-165463
Seq: aagaaacgagtcaagaaccaacagtag------ataatcaaaaagagcaa...
ID: Bacillus_cereus_G9842.gbk/155864-159582
Seq: aagaaactagtcaagaaccaacagtagataaccataatcaaaaagagcaa...
ID: Bacillus_cereus_Q1.gbk/155803-159513
Seq: aagaaactagccaagagccaaca

In [12]:
record.id

'Bacillus_weihenstephanensis_KBAB4.gbk/161684-165393'

In [20]:
record.annotations.get('start')

161684

In [29]:
cleaned_block = remove_gap_columns(first_block)

In [30]:
cleaned_block.get_alignment_length()

3706